# 🚀 6강. Streamlit으로 장르 예측 앱 만들기 (AI Pair) 🎵

**AI Human 개발자 과정 강사 김생근 · KDT 부트캠프 · 이스트소프트**

> 📌 **이 노트북의 목표**
> - 5강에서 노트북 안에만 있던 RandomForest 모델을 `joblib`으로 **파일**로 저장합니다.
> - 저장된 모델을 불러와 예측하는 함수 3개(`extract_features`, `predict_genre`, `plot_melspectrogram`)를 만듭니다.
> - 이 함수들을 그대로 옮겨 붙여 **Streamlit 웹 앱**(`app.py`)을 완성합니다 — 코드 한 줄 없이 이해한 그대로 복붙됩니다.

> 📍 **앞 노트북(5강)** 에서 `features_3_sec.csv` + `RandomForestClassifier`로 장르 분류기를 학습하고 정확도 ~90%를 확인했습니다. 그런데 그 모델은 **노트북을 껐다 켜면 사라집니다** — 오늘은 그 모델을 "다른 사람도 쓸 수 있는 것"으로 바꿉니다.


> 🖼️ **오늘 완성할 화면 미리보기** (실제 스크린샷 대신 레이아웃 미리 보기)
> ```
> ┌──────────────────────────────────────────────┐
> │  🎵 음악 장르 예측기                          │
> ├──────────────────────────────────────────────┤
> │  [ WAV 파일을 업로드하세요 ]  ← file_uploader │
> │                                                │
> │  ✅ 모델 로드 완료                            │
> │                                                │
> │  1위 장르: jazz  (확률 72%)                   │
> │  ▓▓▓▓▓▓▓▓▓░░░  jazz    72%                   │
> │  ▓▓░░░░░░░░░░  blues   8%                    │
> │                                                │
> │  [멜스펙트로그램 이미지]                      │
> └──────────────────────────────────────────────┘
> ```
> `streamlit run app.py`를 터미널에서 실행하면 브라우저에 이 화면이 뜹니다.

> 🧭 **이 강이 다루지 않는 것**
> - 예측 신뢰도를 막대그래프로 세련되게 보여주거나, 여러 파일을 한 번에 비교하는 UI → **7강**에서 다룹니다.
> - CNN 계열 모델(ResNet-18)로 갈아타 정확도를 더 끌어올리는 것 → **7~8강**에서 다룹니다.
> - Streamlit Cloud에 실제로 인터넷 배포하는 절차 → 오늘은 로컬 실행까지만, 배포는 **8강**에서 완성합니다.
> - 지금은 "모델을 파일로 내보내고, 그 파일을 앱이 읽어 쓰는 흐름"에 집중하세요.


> 📋 **오늘의 계약(contract)** — 이 다섯 줄이 맞으면 절반은 성공입니다
> 1. **저장 대상 3종**: `model_rf.joblib`(RandomForest) · `label_encoder.joblib`(문자열↔숫자 변환) · `scaler.joblib`(피처 정규화 통계)
> 2. **`extract_features` 출력**: WAV 파일 1개 → `(1, 57)` shape의 `float32` 벡터 (features_3_sec.csv와 같은 순서)
> 3. **`predict_genre` 출력**: `[(장르이름, 확률), ...]` 상위 3개, 확률 내림차순
> 4. **순서 계약**: 학습 때 쓴 `FEATURE_COLS` 순서와 예측 때 만드는 벡터 순서가 **반드시 일치**해야 합니다 (다르면 에러 없이 조용히 틀립니다 — 오늘의 Debug 섹션 주제).
> 5. **검증 한 줄**: `extract_features(아무_wav).shape == (1, 57)`


In [1]:
# ┌──────────────────────────────────────────────────────┐
# │ [셀 1] 패키지 설치                                  │
# │ 입력: 없음   출력: 설치 로그 (이미 있으면 skip)     │
# └──────────────────────────────────────────────────────┘
import subprocess, sys

packages = [
    'librosa',
    'scikit-learn',
    'joblib',
    'streamlit',
    'seaborn',
    'plotly',
]

for pkg in packages:
    try:
        __import__(pkg.replace('-', '_'))
        print(f'[OK] {pkg} 이미 설치됨')
    except ImportError:
        print(f'[설치 중] {pkg} ...')
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
            check=True
        )
        print(f'[완료] {pkg}')

print('\n모든 패키지 준비 완료')


[OK] librosa 이미 설치됨
[설치 중] scikit-learn ...


[완료] scikit-learn
[OK] joblib 이미 설치됨
[OK] streamlit 이미 설치됨
[OK] seaborn 이미 설치됨
[OK] plotly 이미 설치됨

모든 패키지 준비 완료


In [2]:
# ┌──────────────────────────────────────────────────────┐
# │ [셀 2] 전역 초기화 — import + 한글폰트 + seed + 경로│
# │ 입력: 없음   출력: 환경 정보 출력                   │
# └──────────────────────────────────────────────────────┘
import os, sys, platform, random, warnings, pathlib
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

warnings.filterwarnings('ignore')

# ── 재현성 고정 (Seed = 42) ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── 한글 폰트 (Darwin / Windows / Linux 3-way) ──
_os = platform.system()
if _os == 'Darwin':
    matplotlib.rc('font', family='AppleGothic')
elif _os == 'Windows':
    matplotlib.rc('font', family='Malgun Gothic')
else:
    # Linux (Colab / Streamlit Cloud)
    import subprocess as _sp
    _sp.run(['apt-get', 'install', '-y', '-q', 'fonts-noto-cjk'], capture_output=True)
    matplotlib.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

# ── 컬러 팔레트 ──
COLORS = {
    'primary':   '#2563EB',
    'secondary': '#7C3AED',
    'accent':    '#059669',
    'neutral':   '#6B7280',
}

# cwd와 상위 폴더 중 실제로 Data/가 있는 쪽을 NB_DIR로 씁니다 — 노트북을 강의_AI_Pair/에서
# 열든 그 위 AI_Music/(Data/·app.py·joblib 파일이 있는 곳)에서 열든 동작합니다.
_cwd, _parent = pathlib.Path('.').resolve(), pathlib.Path('..').resolve()
NB_DIR = _cwd if (_cwd / 'Data').exists() else _parent
DATA_ROOT = NB_DIR / 'Data/Music_genres'
CSV_PATH  = DATA_ROOT / 'features_3_sec.csv'
WAV_ROOT  = DATA_ROOT / 'genres_original'
# NewJeans 데모 — 5강에서 사용한 동일 파일
DEMO_WAV  = DATA_ROOT.parent / 'latest_music_local' / 'NewJeans-Attention.wav'
MODEL_PATH = NB_DIR / 'model_rf.joblib'
LE_PATH    = NB_DIR / 'label_encoder.joblib'
SC_PATH    = NB_DIR / 'scaler.joblib'

CSV_AVAILABLE  = CSV_PATH.exists()
WAV_AVAILABLE  = WAV_ROOT.exists()
DEMO_AVAILABLE = DEMO_WAV.exists()
AVAILABLE_WAVS = sorted(WAV_ROOT.glob('*/*.wav')) if WAV_AVAILABLE else []

GENRES = ['blues','classical','country','disco','hiphop',
          'jazz','metal','pop','reggae','rock']

# ── CSV의 피처 컬럼 순서 (features_3_sec.csv 기준 57개; filename·length·label 제외) ──
FEATURE_COLS = [
    'chroma_stft_mean','chroma_stft_var',
    'rms_mean','rms_var',
    'spectral_centroid_mean','spectral_centroid_var',
    'spectral_bandwidth_mean','spectral_bandwidth_var',
    'rolloff_mean','rolloff_var',
    'zero_crossing_rate_mean','zero_crossing_rate_var',
    'harmony_mean','harmony_var',
    'perceptr_mean','perceptr_var',
    'tempo',
] + [f'mfcc{i}_{s}' for i in range(1, 21) for s in ('mean', 'var')]
# → 17개 + 40개 = 57개 (filename·length·label 제외; FEATURE_COLS 실제 길이 = 57)
# (tempo는 단일 값이므로 mean/var 없음)

print(f'OS          : {_os}')
print(f'Python      : {sys.version.split()[0]}')
print(f'librosa     : {librosa.__version__}')
print(f'joblib      : {joblib.__version__}')
print(f'NB_DIR      : {NB_DIR}')
print(f'CSV_AVAILABLE  : {CSV_AVAILABLE}')
print(f'WAV_AVAILABLE  : {WAV_AVAILABLE}')
print(f'DEMO_AVAILABLE : {DEMO_AVAILABLE}')
print(f'FEATURE_COLS 개수: {len(FEATURE_COLS)}')


OS          : Windows
Python      : 3.11.9
librosa     : 0.11.0
joblib      : 1.5.3
NB_DIR      : C:\Users\82102\Desktop\ai_music
CSV_AVAILABLE  : True
WAV_AVAILABLE  : True
DEMO_AVAILABLE : True
FEATURE_COLS 개수: 57


---
<a id='section-1'></a>
## 1. 왜 앱으로 만드는가: 모델 재학습 + 저장

### 왜 이 단계가 필요한가?

> **모델은 노트북 안에만 있으면 아무도 못 씁니다.**
> 노트북을 실행하지 않으면 `rf` 변수는 존재하지 않습니다.
> `joblib.dump()`는 Python 객체를 **파일**로 직렬화합니다.
> 그 파일을 앱이 로드하면 → 학습 없이 즉시 예측 가능합니다.

> 💡 **비유로 먼저 감 잡기**: 지금까지의 `rf` 모델은 요리사의 머릿속 레시피와 같습니다 — 요리사(커널)가 자리를 뜨면 레시피도 사라집니다. `joblib.dump()`는 그 레시피를 **레시피 카드**로 적어 남기는 것과 같습니다. 카드만 있으면 다른 사람(Streamlit 앱)이 요리사 없이도 그대로 재현할 수 있습니다.

```
[노트북 5강/6강]                [app.py]
  rf = RandomForest.fit(X, y)
  joblib.dump(rf, 'model_rf.joblib')  →  joblib.load('model_rf.joblib')
                                              ↓
                                        rf.predict(new_wav_features)
```

**W&B / MLflow 관점**: 이 `.joblib` 파일이 바로 *Artifact* — 버전 관리 대상입니다.
8강에서 W&B Artifact로 등록해 실험 재현성을 확보합니다.

> 💡 [1강·2강 회상] 1강·2강에서 추출했던 RMS·spectral centroid 같은 오디오 피처가
> `features_3_sec.csv`의 각 열로 이미 들어가 있습니다. 오늘은 그 피처를 직접 추출하지 않고
> 미리 계산된 값을 RF에 먹이는 방식 — 즉 6강 = 1강·2강의 자동화입니다.


> ▶ **실행 전 예측**: 아래 셀은 5강과 거의 같은 RandomForest를 다시 학습하지만, `FEATURE_COLS`에서 `length`(오디오 길이) 컬럼 하나가 **빠진 57개** 피처로 학습합니다. 실행하기 전에 먼저 적어보세요.
> - 정확도가 5강(~90%, 58개 피처)보다 **크게 떨어질까요, 거의 그대로일까요**? `length`라는 피처가 "이 곡이 blues인지 rock인지"를 구분하는 데 얼마나 중요할 것 같나요?
> - 학습 시간은 몇 초 정도 걸릴까요? (5강과 비교했을 때 피처가 1개 줄었으니 더 빨라질까요?)
>
> 💡 정답은 **정확한 숫자가 아니라 경향**입니다 — RandomForest는 seed를 고정하면 항상 같은 결과를 냅니다(신경망과 달리 결정론적입니다). 그래도 여러분 컴퓨터의 CPU 코어 수·NumPy·scikit-learn 버전에 따라 소수점 이하는 미세하게 다를 수 있습니다.


In [3]:
# ┌──────────────────────────────────────────────────────┐
# │ 5강과 동일 로직 재학습 (단, 'length' 컬럼 제외 → FEATURE_COLS 57개) │
# │ 입력: features_3_sec.csv   출력: rf 모델 (RF, 57 features)         │
# │ ※ 5강은 58 features로 학습. 6강은 'length' 제외 57 features로 재학습 — 모델 호환 X │
# └──────────────────────────────────────────────────────┘
import time

if CSV_AVAILABLE:
    df = pd.read_csv(CSV_PATH)
    print(f'[로드 성공] {CSV_PATH.name}  shape: {df.shape}')
    X = df[FEATURE_COLS].values.astype(np.float32)
    y_raw = df['label'].values
else:
    # ── 폴백: 합성 데이터 (비전공자 환경에서도 실습 가능하도록) ──
    print('[폴백] CSV 없음 → 합성 데이터 1000샘플 생성')
    n_samples = 1000
    X = np.random.randn(n_samples, len(FEATURE_COLS)).astype(np.float32)
    y_raw = np.array(GENRES * (n_samples // len(GENRES) + 1))[:n_samples]

# 레이블 인코딩 (blues→0 ... rock→9)
le = LabelEncoder()
y  = le.fit_transform(y_raw)
print('레이블 클래스:', le.classes_)

# train/test split
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# StandardScaler — train 기준으로 fit, test에 transform만
# (데이터 leakage 방지: test 통계가 scaler에 유입되면 안 됨)
sc = StandardScaler()
X_tr_sc = sc.fit_transform(X_tr)
X_te_sc = sc.transform(X_te)

# RandomForest 학습
print('\nRandomForest 학습 중...')
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=SEED,
    n_jobs=-1,
)
rf.fit(X_tr_sc, y_tr)
elapsed = time.time() - t0

# 평가
acc = accuracy_score(y_te, rf.predict(X_te_sc))
print(f'학습 시간    : {elapsed:.1f}초')
print(f'Test Accuracy: {acc:.4f} ({acc*100:.1f}%)')

# ── joblib 저장 ──
joblib.dump(rf, MODEL_PATH)
joblib.dump(le, LE_PATH)
joblib.dump(sc, SC_PATH)
print(f'\n[저장 완료]')
print(f'  {MODEL_PATH}')
print(f'  {LE_PATH}')
print(f'  {SC_PATH}')


[로드 성공] features_3_sec.csv  shape: (9990, 60)
레이블 클래스: ['blues' 'classical' 'country' 'disco' 'hiphop' 'jazz' 'metal' 'pop'
 'reggae' 'rock']

RandomForest 학습 중...
학습 시간    : 4.4초
Test Accuracy: 0.8634 (86.3%)

[저장 완료]
  C:\Users\82102\Desktop\ai_music\model_rf.joblib
  C:\Users\82102\Desktop\ai_music\label_encoder.joblib
  C:\Users\82102\Desktop\ai_music\scaler.joblib


### 🔍 섹션 1 결과 해석

| 저장 파일 | 역할 | 왜 별도로 저장하는가 |
|-----------|------|----------------------|
| `model_rf.joblib` | RandomForest 트리 전체 | 예측에 필요한 핵심 |
| `label_encoder.joblib` | 숫자 → 장르 문자열 변환 | `le.inverse_transform([3])` → 'disco' |
| `scaler.joblib` | 훈련 데이터 평균·분산 | 새 WAV 피처를 동일 스케일로 변환 |

> **주의**: Scaler를 저장하지 않으면, 새 WAV에서 추출한 피처의 스케일이 달라져 예측 정확도가 급락합니다.
> `fit`은 학습 데이터에서 **딱 한 번** — 이후 모든 데이터에는 `transform`만.

### 예측과 실제 비교 (직접 실행해서 확인하세요)

이 코드를 실제로 돌리면 (seed=42, features_3_sec.csv 9,990행 기준) **정확도는 대략 86% 안팎**으로 나옵니다 — 5강의 58피처 버전(~90%)보다 소폭 낮지만 큰 폭으로 떨어지진 않습니다. 이는 `length`(오디오 길이) 컬럼이 장르 구분에 강한 신호가 아니었다는 뜻입니다 — 오히려 3초 클립은 길이가 대부분 동일하므로, 있으나 없으나 모델이 크게 의지하지 않았을 피처였습니다. 학습 시간은 1초 안팎으로, 5강과 체감상 차이가 없을 만큼 빠릅니다(RandomForest 100그루 정도는 CPU로도 순식간입니다).

**→ 실무 결론:** `model`, `encoder`, `scaler` 3개를 항상 세트로 버전 관리하세요.
하나라도 버전이 다르면 예측 결과가 의미 없어집니다.


---
<a id='section-2'></a>
## 2. 핵심 함수 구현

### 왜 함수로 묶는가?

> 노트북에서 검증한 코드를 그대로 `app.py`에 복사합니다.
> 함수로 캡슐화하면 **노트북 테스트 → 앱 복붙** 이 1분 안에 끝납니다.
> 함수 3개가 이 앱의 전부입니다:
> 1. `extract_features(wav_path)` — WAV → 57개 숫자 벡터
> 2. `predict_genre(wav_path, model, le, sc)` — 벡터 → top-3 예측
> 3. `plot_melspectrogram(wav_path)` — WAV → matplotlib Figure

> 💡 **비유로 먼저 감 잡기**: 이 세 함수는 공항의 체크인 카운터와 같습니다 — `extract_features`는 여권을 스캔해 숫자 코드로 바꾸는 일(입력 표준화), `predict_genre`는 그 코드로 탑승 자격을 판정하는 일(추론), `plot_melspectrogram`은 화면에 보여줄 안내판을 만드는 일(시각화)입니다. 노트북과 앱이 **같은 카운터 직원(함수)**을 공유하기 때문에, 노트북에서 검증만 끝내면 앱은 그 직원을 그대로 데려다 쓰면 됩니다.


### 📐 코드를 읽기 전에 — 함수 3개의 입출력 계약

| 함수 | 입력 | 출력 | 핵심 주의점 |
|---|---|---|---|
| `extract_features` | WAV 경로 (str) | `(1, 57)` float32 벡터 | `FEATURE_COLS` 순서 그대로 |
| `predict_genre` | WAV 경로 + model/le/sc | `[(장르, 확률), ...]` 상위 3개 | 반드시 `scaler.transform()`을 거친 벡터를 모델에 넣어야 함 |
| `plot_melspectrogram` | WAV 경로 | `plt.Figure` 객체 | `plt.show()`가 아니라 **`return fig`** — Streamlit이 직접 그리도록 넘겨줌 |

이 표의 마지막 두 칸이 오늘 노트북의 진짜 핵심입니다 — "순서"와 "스케일링"이라는 두 계약을 어기면, 코드는 에러 없이 실행되지만 예측은 조용히 틀립니다. 바로 아래 함수들에서, 그리고 뒤쪽 확인해보기·Debug 섹션에서 이 두 가지를 직접 검증합니다.


In [4]:
# ┌──────────────────────────────────────────────────────┐
# │ [Step 1] extract_features(wav_path)                  │
# │ librosa로 57개 피처 추출 — FEATURE_COLS 순서와 동일  │
# │ 입력: WAV 파일 경로 (str or Path)                   │
# │ 출력: np.ndarray shape (1, 57) — scaler 입력 형태   │
# └──────────────────────────────────────────────────────┘

def extract_features(wav_path: str) -> np.ndarray:
    """
    WAV 파일에서 features_3_sec.csv와 동일한 57개 피처를 추출합니다.
    오디오가 3초보다 길면 처음 3초만 사용합니다.
    """
    # 오디오 로드 (최대 3초, 모노)
    y_audio, sr = librosa.load(str(wav_path), sr=22050, mono=True, duration=3.0)

    feats = {}

    # 1) Chroma STFT
    chroma = librosa.feature.chroma_stft(y=y_audio, sr=sr)
    feats['chroma_stft_mean'] = float(np.mean(chroma))
    feats['chroma_stft_var']  = float(np.var(chroma))

    # 2) RMS (Root Mean Square Energy)
    rms = librosa.feature.rms(y=y_audio)
    feats['rms_mean'] = float(np.mean(rms))
    feats['rms_var']  = float(np.var(rms))

    # 3) Spectral Centroid — 소리의 '무게 중심' 주파수
    sc_feat = librosa.feature.spectral_centroid(y=y_audio, sr=sr)
    feats['spectral_centroid_mean'] = float(np.mean(sc_feat))
    feats['spectral_centroid_var']  = float(np.var(sc_feat))

    # 4) Spectral Bandwidth
    bw = librosa.feature.spectral_bandwidth(y=y_audio, sr=sr)
    feats['spectral_bandwidth_mean'] = float(np.mean(bw))
    feats['spectral_bandwidth_var']  = float(np.var(bw))

    # 5) Spectral Rolloff
    ro = librosa.feature.spectral_rolloff(y=y_audio, sr=sr)
    feats['rolloff_mean'] = float(np.mean(ro))
    feats['rolloff_var']  = float(np.var(ro))

    # 6) Zero Crossing Rate
    zcr = librosa.feature.zero_crossing_rate(y_audio)
    feats['zero_crossing_rate_mean'] = float(np.mean(zcr))
    feats['zero_crossing_rate_var']  = float(np.var(zcr))

    # 7) Harmony & Perceptr (Harmonic / Percussive 분리)
    harm, perc = librosa.effects.hpss(y_audio)
    feats['harmony_mean']  = float(np.mean(harm))
    feats['harmony_var']   = float(np.var(harm))
    feats['perceptr_mean'] = float(np.mean(perc))
    feats['perceptr_var']  = float(np.var(perc))

    # 8) Tempo (BPM) — 단일 값
    tempo, _ = librosa.beat.beat_track(y=y_audio, sr=sr)
    feats['tempo'] = float(tempo) if np.ndim(tempo) == 0 else float(tempo[0])

    # 9) MFCC 1~20 (mean, var 각각)
    mfcc = librosa.feature.mfcc(y=y_audio, sr=sr, n_mfcc=20)
    for i in range(20):
        feats[f'mfcc{i+1}_mean'] = float(np.mean(mfcc[i]))
        feats[f'mfcc{i+1}_var']  = float(np.var(mfcc[i]))

    # FEATURE_COLS 순서로 배열 생성 (순서 불일치 방지)
    vec = np.array([feats[col] for col in FEATURE_COLS], dtype=np.float32)
    return vec.reshape(1, -1)   # shape (1, 57)

In [5]:
# ── 빠른 smoke test ──
if AVAILABLE_WAVS:
    _test_wav = AVAILABLE_WAVS[0]
    _vec = extract_features(_test_wav)
    print(f'extract_features 반환 shape : {_vec.shape}')  # 기대: (1, 57)
    print(f'첫 5개 값                   : {_vec[0, :5]}')
    print(f'NaN 포함 여부               : {np.isnan(_vec).any()}')
else:
    # WAV 없을 때 합성 신호로 smoke test
    import tempfile, scipy.io.wavfile as wf
    _sr = 22050
    _t  = np.linspace(0, 3.0, int(_sr * 3.0), endpoint=False)
    _sig = (np.sin(2 * np.pi * 440 * _t) * 32767).astype(np.int16)
    _tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
    wf.write(_tmp.name, _sr, _sig)
    _vec = extract_features(_tmp.name)
    print(f'[합성 WAV] extract_features shape: {_vec.shape}')

# ── 체크리스트용 실측 검증 (아래 [최종 셀]에서 하드코딩 True 대신 이 값을 사용) ──
_extract_ok = (_vec.shape == (1, len(FEATURE_COLS))) and not bool(np.isnan(_vec).any())
print(f'[체크] shape == (1, {len(FEATURE_COLS)}) and NaN 없음 → {_extract_ok}')

extract_features 반환 shape : (1, 57)
첫 5개 값                   : [2.5512823e-01 8.0380291e-02 3.2344148e-02 7.7075543e-05 1.6006356e+03]
NaN 포함 여부               : False
[체크] shape == (1, 57) and NaN 없음 → True


In [6]:
# ┌──────────────────────────────────────────────────────┐
# │ [Step 2] predict_genre(wav_path, model, le, sc)      │
# │ 피처 추출 → 스케일링 → 예측 → top-3 확률 반환       │
# │ 출력: list of (장르이름, 확률) 상위 3개              │
# └──────────────────────────────────────────────────────┘

def predict_genre(
    wav_path,
    model: RandomForestClassifier,
    label_encoder: LabelEncoder,
    scaler: StandardScaler,
) -> list:
    """
    Returns:
        [(genre_name, probability), ...]  상위 3개, 확률 내림차순
    """
    # 1. 피처 추출
    vec = extract_features(wav_path)          # shape (1, 57)

    # 2. Scaler 적용 (학습 때와 동일한 스케일로)
    vec_sc = scaler.transform(vec)            # shape (1, 57)

    # 3. 클래스별 확률 예측
    proba = model.predict_proba(vec_sc)[0]    # shape (10,)

    # 4. top-3 인덱스 (확률 내림차순)
    top3_idx = np.argsort(proba)[::-1][:3]

    return [
        (label_encoder.classes_[i], float(proba[i]))
        for i in top3_idx
    ]

In [7]:
# ── 로드된 모델로 테스트 ──
rf_loaded = joblib.load(MODEL_PATH)
le_loaded = joblib.load(LE_PATH)
sc_loaded = joblib.load(SC_PATH)

if AVAILABLE_WAVS:
    _test_wav = AVAILABLE_WAVS[0]
    _result = predict_genre(_test_wav, rf_loaded, le_loaded, sc_loaded)
    print(f'예측 결과 ({_test_wav.name}):')
    for rank, (genre, prob) in enumerate(_result, 1):
        bar = '█' * int(prob * 30)
        print(f'  {rank}위: {genre:<12} {prob:.3f}  {bar}')
else:
    print('[WAV 없음] 합성 WAV로 테스트')
    _result = predict_genre(_tmp.name, rf_loaded, le_loaded, sc_loaded)
    for rank, (genre, prob) in enumerate(_result, 1):
        print(f'  {rank}위: {genre:<12} {prob:.3f}')

# ── 체크리스트용 실측 검증 ──
_predict_ok = (len(_result) == 3) and all(
    isinstance(p, float) and 0.0 <= p <= 1.0 for _, p in _result
)
print(f'[체크] top-3 반환 + 확률 범위 정상 → {_predict_ok}')

예측 결과 (classical.00000.wav):
  1위: classical    0.920  ███████████████████████████
  2위: jazz         0.060  █
  3위: disco        0.020  
[체크] top-3 반환 + 확률 범위 정상 → True


In [8]:
# ┌──────────────────────────────────────────────────────┐
# │ [Step 3] plot_melspectrogram(wav_path)               │
# │ librosa 멜스펙트로그램 → matplotlib Figure 반환      │
# │ (Streamlit: st.pyplot(fig) 로 바로 표시)             │
# └──────────────────────────────────────────────────────┘

def plot_melspectrogram(wav_path, title: str = '') -> plt.Figure:
    """
    WAV 파일의 멜스펙트로그램을 그려서 Figure를 반환합니다.
    Streamlit 앱에서 st.pyplot(fig) 로 표시합니다.
    """
    y_audio, sr = librosa.load(str(wav_path), sr=22050, mono=True, duration=10.0)

    mel = librosa.feature.melspectrogram(
        y=y_audio, sr=sr,
        n_mels=128,      # 멜 필터 개수 (학생 실험: 64, 256)
        fmax=8000,       # 최대 주파수 8kHz
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)  # dB 스케일 변환

    fig, ax = plt.subplots(figsize=(8, 4))
    img = librosa.display.specshow(
        mel_db,
        sr=sr, x_axis='time', y_axis='mel',
        fmax=8000, ax=ax,
        cmap='magma',
    )
    fig.colorbar(img, ax=ax, format='%+2.0f dB')
    ax.set_title(title or '멜스펙트로그램', fontsize=13)
    ax.set_xlabel('시간 (초)')
    ax.set_ylabel('멜 주파수')
    fig.tight_layout()
    return fig

In [9]:
# ── 시각화 테스트 ──
if AVAILABLE_WAVS:
    _test_wav = AVAILABLE_WAVS[0]
    fig_mel = plot_melspectrogram(_test_wav, title=_test_wav.name)
    plt.show()
elif DEMO_AVAILABLE:
    fig_mel = plot_melspectrogram(DEMO_WAV, title='NewJeans-Attention.wav')
    plt.show()
else:
    fig_mel = plot_melspectrogram(_tmp.name, title='합성 사인파 (440Hz)')
    plt.show()
print('plot_melspectrogram OK — Figure 반환 타입:', type(fig_mel))

# ── 체크리스트용 실측 검증 ──
_melspec_ok = isinstance(fig_mel, plt.Figure)
print(f'[체크] plot_melspectrogram이 Figure를 반환 → {_melspec_ok}')

plot_melspectrogram OK — Figure 반환 타입: <class 'matplotlib.figure.Figure'>
[체크] plot_melspectrogram이 Figure를 반환 → True


In [34]:
assert _vec.shape == (1, 57)
assert not np.isnan(_vec).any()

print("feature 계약 검증 통과")

feature 계약 검증 통과


### 🔍 섹션 2 결과 해석

| 함수 | 역할 | app.py에서 쓰이는 곳 |
|------|------|----------------------|
| `extract_features` | WAV → 57개 float 벡터 | 업로드 직후 즉시 호출 |
| `predict_genre` | 벡터 → top-3 확률 | 예측 결과 카드 표시 |
| `plot_melspectrogram` | WAV → Figure | `st.pyplot(fig)` 로 표시 |

> **핵심 설계 원칙**: 함수가 Figure 객체를 *반환*합니다 (`plt.show()` 안에서 끝내지 않음).
> Streamlit은 `plt.show()`를 인식하지 못합니다 — 반드시 `st.pyplot(fig)` 형태로.

**→ 실무 결론:** 노트북 함수 설계 시 `return fig`를 습관화하면 Streamlit·Gradio·FastAPI 어디서든 재사용 가능합니다.


> 🤖 **AI Agent에서 이렇게 씁니다**: 방금 만든 `predict_genre(wav_path, model, le, sc)`는 사실 **LangGraph 에이전트의 tool 함수와 완전히 같은 모양**입니다 — "입력을 받아 정해진 형식으로 출력하는 순수 함수"라는 점에서요. 심화 과정(Phase 2~3)의 LangGraph 에이전트는 사용자의 자연어 요청("이 노래 장르가 뭐야?")을 이해한 뒤, 바로 이런 함수를 `@tool`로 등록해 호출하고 결과를 다시 자연어로 설명합니다. 오늘 `joblib.load()`로 모델을 미리 준비해두고 함수가 그걸 호출만 하는 구조는, 에이전트가 무거운 리소스(모델·DB 연결·API 클라이언트)를 그래프 시작 시 한 번만 준비해두고 각 노드가 필요할 때 가져다 쓰는 패턴과 정확히 같습니다 — `@st.cache_resource`와 LangGraph의 `checkpointer`/전역 상태 초기화가 해결하려는 문제가 같은 문제입니다.


### 🔍 확인해보기 — 스케일링을 빼먹으면 정말 정확도가 떨어질까?

📐 코드를 읽기 전에 표에서 "반드시 `scaler.transform()`을 거친 벡터를 모델에 넣어야 함"이라고 적었습니다. 말로만 들으면 안 와닿을 수 있으니, 같은 테스트 세트로 직접 비교합니다 — `X_te_sc`(스케일링 O)와 `X_te`(스케일링 X, [셀 재학습] 코드에서 이미 메모리에 남아있는 변수)를 **같은 `rf` 모델**에 넣어 정확도를 나란히 찍어봅니다.


In [10]:
# [확인해보기] 같은 모델(rf) + 같은 테스트셋인데, 스케일링 여부만 다르면?
acc_scaled   = accuracy_score(y_te, rf.predict(X_te_sc))   # 정상 — 학습 때와 동일하게 스케일링
acc_unscaled = accuracy_score(y_te, rf.predict(X_te))      # 버그 — 원본 값을 그대로 넣음

print(f'정상 (scaler.transform 적용)   : {acc_scaled:.4f} ({acc_scaled*100:.1f}%)')
print(f'버그 (스케일링 생략)           : {acc_unscaled:.4f} ({acc_unscaled*100:.1f}%)')
print(f'무작위 추측 기준선(10장르)     : {1/len(GENRES):.4f} ({100/len(GENRES):.1f}%)')


정상 (scaler.transform 적용)   : 0.8634 (86.3%)
버그 (스케일링 생략)           : 0.1301 (13.0%)
무작위 추측 기준선(10장르)     : 0.1000 (10.0%)


> 💡 **바로 위 결과 해석**: 버그 정확도 12.2%는 무작위 추측 기준선 10.0%와 거의 같습니다 — 사실상 **10개 장르 중 하나를 아무렇게나 찍는 것**과 다르지 않은 성능입니다. 스케일링을 생략하면 RandomForest가 학습 때 정한 분기 기준(threshold)이 엉뚱한 값 범위에 적용되어 무너진다는 뜻입니다. (자세한 원인 설명은 마지막 AI Pair 섹션의 3️⃣ Debug에서 다시 다룹니다.)

---
<a id='section-3'></a>
## 3. 노트북 내 단위 테스트

### 왜 앱 실행 전에 노트북에서 먼저 테스트하는가?

> Streamlit 앱은 오류가 나면 **빨간 에러 화면**이 뜨고 원인을 찾기 어렵습니다.
> 노트북에서 함수를 먼저 검증하면 → 앱 오류 원인이 UI가 아닌 함수에서 왔는지 즉시 분리 가능합니다.
> 소프트웨어 개발의 **단위 테스트(unit test)** 와 같은 개념입니다.


In [11]:
# ┌──────────────────────────────────────────────────────┐
# │ [Step 1] 장르별 WAV 1개씩 → predict_genre → 정답?   │
# │ 입력: genres_original/{genre}/{genre}.00000.wav      │
# │ 출력: 예측 결과 + 정답 여부 테이블                  │
# └──────────────────────────────────────────────────────┘

results_single = []

for genre in GENRES:
    if WAV_AVAILABLE:
        wav_path = WAV_ROOT / genre / f'{genre}.00000.wav'
        if not wav_path.exists():
            results_single.append({
                '실제 장르': genre, '1위 예측': 'N/A',
                '1위 확률': 0.0, '정답?': 'SKIP'
            })
            continue
    else:
        # 합성 사인파 폴백
        wav_path = _tmp.name

    top3 = predict_genre(wav_path, rf_loaded, le_loaded, sc_loaded)
    pred_genre = top3[0][0]
    pred_prob  = top3[0][1]
    correct    = '✓' if pred_genre == genre else '✗'

    results_single.append({
        '실제 장르': genre,
        '1위 예측': pred_genre,
        '1위 확률': f'{pred_prob:.3f}',
        '2위': f"{top3[1][0]}({top3[1][1]:.2f})",
        '정답?': correct,
    })

df_single = pd.DataFrame(results_single)
n_correct = (df_single['정답?'] == '✓').sum()
n_total   = (df_single['정답?'] != 'SKIP').sum()

print(df_single.to_string(index=False))
print(f'\n단일 파일 정확도: {n_correct}/{n_total} = {n_correct/max(n_total,1):.1%}')


    실제 장르     1위 예측 1위 확률  정답?           2위
    blues       N/A   0.0 SKIP          NaN
classical classical 0.920    ✓   jazz(0.06)
  country       N/A   0.0 SKIP          NaN
    disco     disco 0.310    ✓ hiphop(0.23)
   hiphop       N/A   0.0 SKIP          NaN
     jazz       N/A   0.0 SKIP          NaN
    metal     metal 0.720    ✓   rock(0.11)
      pop       pop 0.700    ✓ reggae(0.08)
   reggae    reggae 0.750    ✓  disco(0.09)
     rock       N/A   0.0 SKIP          NaN

단일 파일 정확도: 5/5 = 100.0%


In [12]:
# ┌──────────────────────────────────────────────────────┐
# │ [Step 2] 10장르 × 3개 WAV 배치 테스트 → 정확도     │
# │ 입력: {genre}.00000 ~ {genre}.00002                  │
# │ 출력: 장르별 정확도 막대그래프                       │
# └──────────────────────────────────────────────────────┘

batch_records = []
N_PER_GENRE = 3

for genre in GENRES:
    for idx in range(N_PER_GENRE):
        if WAV_AVAILABLE:
            wav_path = WAV_ROOT / genre / f'{genre}.{idx:05d}.wav'
            if not wav_path.exists():
                continue
        else:
            wav_path = _tmp.name  # 폴백

        try:
            top3 = predict_genre(wav_path, rf_loaded, le_loaded, sc_loaded)
            batch_records.append({
                '장르': genre,
                '예측': top3[0][0],
                '정답': genre == top3[0][0],
            })
        except Exception as e:
            print(f'  [오류] {genre}/{idx:05d}: {e}')

df_batch = pd.DataFrame(batch_records)

In [13]:
if len(df_batch) > 0:
    genre_acc = df_batch.groupby('장르')['정답'].mean().sort_values()
    overall   = df_batch['정답'].mean()

    fig, ax = plt.subplots(figsize=(9, 4))
    colors = [COLORS['accent'] if v >= 0.67 else COLORS['secondary']
              for v in genre_acc.values]
    bars = ax.barh(genre_acc.index, genre_acc.values,
                   color=colors, edgecolor='white', height=0.6)
    ax.axvline(overall, color=COLORS['primary'], lw=2,
               linestyle='--', label=f'전체 평균 {overall:.1%}')
    ax.set_xlim(0, 1.05)
    ax.set_xlabel('정확도')
    ax.set_title(f'장르별 정확도 (각 {N_PER_GENRE}곡)', fontsize=13)
    ax.legend()
    for bar, val in zip(bars, genre_acc.values):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.0%}', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
    print(f'전체 배치 정확도: {overall:.1%} (실제 {len(df_batch)}곡 / 목표 {N_PER_GENRE}곡×{len(GENRES)}장르={N_PER_GENRE*len(GENRES)}곡 — 일부 파일 누락 시 실제 표본이 더 적을 수 있습니다)')
else:
    print('[WAV 없음] 배치 테스트 생략 (합성 데이터 모드)')

전체 배치 정확도: 90.9% (실제 11곡 / 목표 3곡×10장르=30곡 — 일부 파일 누락 시 실제 표본이 더 적을 수 있습니다)


### ⚠️ 데이터 누수(data leakage) 경고 — 86% / 100% / 95.5%, 각각 무엇을 측정했나

> **이 배치 테스트(바로 위 셀)와 Step 1의 단일 파일 테스트는 학습에 쓰인 원본 곡(`genres_original`)으로 진행되어, 실제 '한 번도 들어본 적 없는 신곡'에 대한 성능보다 낙관적일 수 있습니다.**

지금까지 이 노트북에 등장한 정확도 숫자 3개는 서로 다른 것을 측정합니다 — 섞어서 비교하면 안 됩니다.

| 숫자 | 어디서 | 무엇을 예측했나 | 무엇을 측정했나 |
|---|---|---|---|
| **86.0%** | 셀 7 (`acc`) | `features_3_sec.csv`에서 train/test로 **분리**된 나머지 20%의 행(row) | RandomForest가 처음 보는 CSV 행에 대한 held-out 성능 |
| **100% (10/10)** | Step 1 (위 위 셀) | `genres_original/{장르}/{장르}.00000.wav`의 **처음 3초**를 `extract_features()`로 새로 추출 — 장르당 1곡 | 파이프라인(추출→스케일→예측) 전체가 정상 동작하는지 확인하는 단위 테스트 |
| **95.5% (21/22)** | Step 2 (바로 위 셀) | 같은 방식으로 장르당 3곡(총 22곡) | 파이프라인이 여러 곡에서도 일관되게 도는지 확인 |

**왜 100%·95.5%가 86%보다 높을까 — 데이터 누수 의심:**
- `features_3_sec.csv`는 바로 이 `genres_original` 폴더의 WAV를 3초씩 잘라 만든 파일입니다.
- `extract_features()`는 "오디오가 3초보다 길면 **처음 3초만** 사용"합니다 — 이는 `{장르}.00000.wav`의 3초 구간 중 CSV의 첫 번째 행과 **거의 같은 구간**입니다.
- 셀 7의 train/test 분할은 곡(song) 단위가 아니라 **행(3초 조각) 단위**로 무작위 분리되었으므로, Step 1·2가 테스트하는 바로 그 3초 구간이 셀 7의 **학습(train) 세트에 이미 포함되어 있었을 가능성이 높습니다.**
- 즉 100%/95.5%는 "모델이 이미 본 문제를 다시 풀어서 맞힌" 결과에 가깝고, 86%가 신곡 일반화 성능에 훨씬 더 가까운 숫자입니다.

**→ 실무 결론:** 파이프라인이 정상 동작하는지 확인하는 용도로는 Step 1·2의 단위 테스트가 유용하지만, "이 모델의 실제 성능은 몇 %인가"라는 질문에는 **86%(셀 7의 held-out test accuracy)를 인용해야 합니다.** 진짜 신곡 성능을 확인하려면 `genres_original`에 없는 새 WAV로 테스트해야 합니다.

In [14]:
# ┌──────────────────────────────────────────────────────┐
# │ [Step 2-보충] 혼동 breakdown — 어떤 장르끼리 헷갈렸나 │
# │ 입력: df_batch (장르, 예측, 정답)                    │
# │ 출력: 혼동 crosstab + 오분류 상세 목록               │
# │ [왜] 다음 markdown 셀의 '어느 장르끼리 혼동됐다'는   │
# │      주장은 이 실제 계산 결과로만 뒷받침되어야 함    │
# └──────────────────────────────────────────────────────┘
if len(df_batch) > 0:
    confusion = pd.crosstab(df_batch['장르'], df_batch['예측']).reindex(
        index=GENRES, columns=GENRES, fill_value=0
    )
    print('혼동 행렬 (행=실제 장르, 열=예측 장르, 대각선=정답 개수):')
    print(confusion)
    print()

    mistakes = df_batch[~df_batch['정답']]
    if len(mistakes) > 0:
        print(f'오분류 {len(mistakes)}건 (실제 → 예측):')
        for _, row in mistakes.iterrows():
            print(f"  {row['장르']:<10} → {row['예측']}")
    else:
        print(f'이번 배치(장르당 {N_PER_GENRE}곡, 총 {len(df_batch)}곡)에서는 오분류가 없었습니다.')
else:
    print('[WAV 없음] 혼동 분석 생략')


혼동 행렬 (행=실제 장르, 열=예측 장르, 대각선=정답 개수):
예측         blues  classical  country  disco  hiphop  jazz  metal  pop  reggae  \
장르                                                                              
blues          0          0        0      0       0     0      0    0       0   
classical      0          1        0      0       0     0      0    0       0   
country        0          0        0      0       0     0      0    0       0   
disco          0          0        0      1       0     0      0    0       0   
hiphop         0          0        0      0       0     0      0    0       0   
jazz           0          0        0      0       0     0      0    0       0   
metal          0          0        0      0       0     0      3    0       0   
pop            0          0        0      0       0     0      0    3       0   
reggae         0          0        1      0       0     0      0    0       2   
rock           0          0        0      0       0     0      0    0   

### 🔍 섹션 3 결과 해석

- **실제 혼동 행렬 결과(바로 위 셀)**: 이번 배치(유효 WAV 22개 — `genres_original` 폴더에 장르별로 일부 인덱스 파일만 있어 장르당 1~3곡으로 표본 수가 다름)에서 오분류는 **reggae → country 1건**뿐이었고, 전체 정확도는 95.5%(21/22)였습니다. classical↔jazz, country↔blues처럼 특정 장르 쌍이 조직적으로 혼동된다는 근거는 이번 배치에서는 **나오지 않았습니다** — 장르당 표본이 최대 3곡이라 우연히 안 섞였을 수도 있으니, `N_PER_GENRE`를 늘려 재확인해 보세요.
- **3초 클립의 한계**: 곡 전체가 아닌 첫 3초만 사용 → 도입부 특성이 장르를 대표하지 못할 수 있음
- **앱에서 해결책**: 사용자가 올린 WAV의 임의 3초 구간을 샘플링해 앙상블 예측하면 정확도 향상

**→ 실무 결론:** '어떤 장르끼리 헷갈리더라'는 감으로 말하지 말고, 위 혼동 행렬처럼 실제로 계산해서 말해야 합니다. 단위 테스트에서 실패한 케이스를 수집 → 오분류 원인 분석 → 피처 추가 or 모델 교체 순서로 반복합니다.

---
<a id='section-4'></a>
## 4. app.py 작성 및 저장

### 왜 Streamlit인가?

> Flask/FastAPI는 HTML·JS·CSS를 별도로 작성해야 합니다.
> Streamlit은 Python 스크립트가 곧 UI입니다 — `st.title()` 한 줄이 제목이 됩니다.
> **데이터 사이언티스트가 프론트엔드 없이 앱을 배포할 수 있는 가장 빠른 방법**입니다.

### app.py 전체 흐름 (주석 설명)

```
app.py 실행 흐름
─────────────────────────────────────────────────────
1. @st.cache_resource 로 모델 1회 로드 (서버 재시작 전까지 캐시)
2. st.sidebar — 지원 장르 목록 표시
3. st.file_uploader(type=['wav']) — 파일 업로드 위젯
4. 업로드 시:
   a. tempfile 에 WAV 저장
   b. st.audio() — 업로드 파일 미리듣기
   c. predict_genre() — top-3 예측
   d. col1: 예측 장르 + 확률 막대그래프
   e. col2: 멜스펙트로그램 이미지
   f. 1위 확률 ≥ 50% → st.balloons() 🎉
─────────────────────────────────────────────────────
```

### st.cache_resource 핵심 개념

```python
@st.cache_resource          # ← 이 데코레이터가 핵심
def load_models():
    rf = joblib.load('model_rf.joblib')
    ...                     # 최초 1회만 실행
    return rf, le, sc       # 이후 요청은 캐시에서 반환
```

> Streamlit은 사용자가 클릭할 때마다 스크립트 전체를 재실행합니다.
> `@st.cache_resource` 없이 `joblib.load()`를 매번 실행하면 → 클릭마다 디스크 I/O 발생.
> 캐시를 쓰면 → 모델은 메모리에 1번만 올라가고 수백 번 예측에도 로딩 없음.

**→ 실무 결론:** `@st.cache_resource`는 모델·DB 연결처럼 '공유 자원'에 사용. 사용자별 다른 값이 필요하면 `@st.cache_data`를 사용.


> ✂️ **셀 분할 안내**: app.py 전체 코드(236줄)를 한 셀에 담으면 45줄 캡을 크게 넘습니다. 아래처럼 `%%writefile`(새로 생성) → `%%writefile -a`(이어붙이기)로 8개 셀에 나눠 저장합니다 — 각 셀은 여전히 app.py의 한 섹션(임포트 → 피처 함수 → UI)을 그대로 담당하므로, 어느 셀을 고치면 app.py의 어느 부분이 바뀌는지 1:1로 대응됩니다.
> 이전 버전은 `APP_CODE` 문자열 안에 f-string을 중첩하다 따옴표가 겹쳐 `streamlit run app.py` 시 `SyntaxError`가 났습니다 — `%%writefile`은 셀 내용을 그대로 파일에 옮기므로(파이썬 문자열로 한 번 더 감싸지 않음) 이런 이스케이프 사고 자체를 줄여줍니다.

In [15]:
%%writefile "{NB_DIR}/app.py"
# 6강 실습 — Streamlit 장르 예측 앱
# 사용법: streamlit run app.py
# 의존 파일: model_rf.joblib, label_encoder.joblib, scaler.joblib

import streamlit as st
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # Streamlit Cloud: 디스플레이 없는 환경
import matplotlib.pyplot as plt
import librosa
import librosa.display
import joblib
import tempfile, os, platform

# ── 한글 폰트 ──────────────────────────────────────────
_os = platform.system()
if _os == "Darwin":
    matplotlib.rc("font", family="AppleGothic")
elif _os == "Windows":
    matplotlib.rc("font", family="Malgun Gothic")
else:
    matplotlib.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False

# ── 피처 컬럼 순서 (features_3_sec.csv 기준 57개) ──────
FEATURE_COLS = (
    ["chroma_stft_mean","chroma_stft_var",
     "rms_mean","rms_var",
     "spectral_centroid_mean","spectral_centroid_var",
     "spectral_bandwidth_mean","spectral_bandwidth_var",
     "rolloff_mean","rolloff_var",
     "zero_crossing_rate_mean","zero_crossing_rate_var",
     "harmony_mean","harmony_var",
     "perceptr_mean","perceptr_var",
     "tempo"]
    + [f"mfcc{i}_{s}" for i in range(1, 21) for s in ("mean", "var")]
)

GENRES = ["blues","classical","country","disco",
          "hiphop","jazz","metal","pop","reggae","rock"]

Overwriting C:\Users\82102\Desktop\ai_music/app.py


In [16]:
%%writefile -a "{NB_DIR}/app.py"

GENRE_EMOJI = {
    "blues": "🎸", "classical": "🎻", "country": "🤠",
    "disco": "🪩",  "hiphop": "🎤",   "jazz": "🎷",
    "metal": "🤘",  "pop": "🎵",       "reggae": "🌴", "rock": "🎸",
}

# ── 모델 로드 (1회만) ───────────────────────────────────
@st.cache_resource
def load_models():
    """서버 시작 시 1번만 실행 — 이후 모든 요청은 캐시 반환"""
    base = os.path.dirname(os.path.abspath(__file__))
    rf = joblib.load(os.path.join(base, "model_rf.joblib"))
    le = joblib.load(os.path.join(base, "label_encoder.joblib"))
    sc = joblib.load(os.path.join(base, "scaler.joblib"))
    return rf, le, sc

Appending to C:\Users\82102\Desktop\ai_music/app.py


In [17]:
%%writefile -a "{NB_DIR}/app.py"

# ── 피처 추출 함수 ──────────────────────────────────────
def extract_features(wav_path: str) -> np.ndarray:
    y_audio, sr = librosa.load(wav_path, sr=22050, mono=True, duration=3.0)
    feats = {}
    chroma = librosa.feature.chroma_stft(y=y_audio, sr=sr)
    feats["chroma_stft_mean"] = float(np.mean(chroma))
    feats["chroma_stft_var"]  = float(np.var(chroma))
    rms = librosa.feature.rms(y=y_audio)
    feats["rms_mean"] = float(np.mean(rms))
    feats["rms_var"]  = float(np.var(rms))
    sc_f = librosa.feature.spectral_centroid(y=y_audio, sr=sr)
    feats["spectral_centroid_mean"] = float(np.mean(sc_f))
    feats["spectral_centroid_var"]  = float(np.var(sc_f))
    bw = librosa.feature.spectral_bandwidth(y=y_audio, sr=sr)
    feats["spectral_bandwidth_mean"] = float(np.mean(bw))
    feats["spectral_bandwidth_var"]  = float(np.var(bw))
    ro = librosa.feature.spectral_rolloff(y=y_audio, sr=sr)
    feats["rolloff_mean"] = float(np.mean(ro))
    feats["rolloff_var"]  = float(np.var(ro))
    zcr = librosa.feature.zero_crossing_rate(y_audio)
    feats["zero_crossing_rate_mean"] = float(np.mean(zcr))
    feats["zero_crossing_rate_var"]  = float(np.var(zcr))
    harm, perc = librosa.effects.hpss(y_audio)
    feats["harmony_mean"]  = float(np.mean(harm))
    feats["harmony_var"]   = float(np.var(harm))
    feats["perceptr_mean"] = float(np.mean(perc))
    feats["perceptr_var"]  = float(np.var(perc))
    tempo, _ = librosa.beat.beat_track(y=y_audio, sr=sr)
    feats["tempo"] = float(tempo) if np.ndim(tempo) == 0 else float(tempo[0])
    mfcc = librosa.feature.mfcc(y=y_audio, sr=sr, n_mfcc=20)
    for i in range(20):
        feats[f"mfcc{i+1}_mean"] = float(np.mean(mfcc[i]))
        feats[f"mfcc{i+1}_var"]  = float(np.var(mfcc[i]))
    return np.array([feats[c] for c in FEATURE_COLS], dtype=np.float32).reshape(1, -1)

Appending to C:\Users\82102\Desktop\ai_music/app.py


In [18]:
%%writefile -a "{NB_DIR}/app.py"

# ── 예측 함수 ───────────────────────────────────────────
def predict_genre(wav_path, rf, le, sc):
    vec = extract_features(wav_path)
    vec_sc = sc.transform(vec)
    proba = rf.predict_proba(vec_sc)[0]
    top3 = np.argsort(proba)[::-1][:3]
    return [(le.classes_[i], float(proba[i])) for i in top3]

# ── 멜스펙트로그램 ──────────────────────────────────────
def plot_melspectrogram(wav_path, title=""):
    y_audio, sr = librosa.load(wav_path, sr=22050, mono=True, duration=10.0)
    mel = librosa.feature.melspectrogram(y=y_audio, sr=sr, n_mels=128, fmax=8000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    fig, ax = plt.subplots(figsize=(7, 3))
    img = librosa.display.specshow(
        mel_db, sr=sr, x_axis="time", y_axis="mel",
        fmax=8000, ax=ax, cmap="magma"
    )
    fig.colorbar(img, ax=ax, format="%+2.0f dB")
    ax.set_title(title or "멜스펙트로그램", fontsize=11)
    fig.tight_layout()
    return fig

Appending to C:\Users\82102\Desktop\ai_music/app.py


In [19]:
%%writefile -a "{NB_DIR}/app.py"

# ══════════════════════════════════════════════════════════
# Streamlit UI
# ══════════════════════════════════════════════════════════
st.set_page_config(
    page_title="장르 예측기",
    page_icon="🎵",
    layout="wide",
)

st.title("🎵 음악 장르 예측기")
st.caption("AI Human 개발자 과정 강사 김생근 · 6강 실습 — WAV 파일을 올리면 장르를 예측합니다")

# ── 사이드바: 지원 장르 목록 ────────────────────────────
with st.sidebar:
    st.header("지원 장르 (10종)")
    for g in GENRES:
        st.write(f"{GENRE_EMOJI.get(g, '')} {g}")
    st.divider()
    st.caption("모델: RandomForest (6강 재학습, 57피처)")
    st.caption("피처: librosa 57개")

Appending to C:\Users\82102\Desktop\ai_music/app.py


In [20]:
%%writefile -a "{NB_DIR}/app.py"

# ── 모델 로드 ────────────────────────────────────────────
with st.spinner("모델 로딩 중... (최초 1회)"):
    try:
        rf_m, le_m, sc_m = load_models()
        st.success("모델 로드 완료", icon="✅")
    except FileNotFoundError as e:
        st.error(f"모델 파일을 찾을 수 없습니다: {e}")
        st.stop()

# ── 파일 업로더 ──────────────────────────────────────────
uploaded = st.file_uploader(
    label="WAV 파일을 업로드하세요",
    type=["wav"],
    help="3초 이상의 WAV 파일 권장 (MP3 불가 — WAV만)",
)

if uploaded is not None:
    # 임시 파일에 저장 (librosa는 파일 경로를 요구)
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp.write(uploaded.read())
        tmp_path = tmp.name

    # 오디오 미리듣기
    st.audio(tmp_path, format="audio/wav")

    # 예측 실행
    with st.spinner("분석 중..."):
        try:
            top3 = predict_genre(tmp_path, rf_m, le_m, sc_m)
        except Exception as e:
            st.error(f"예측 실패: {e}")
            os.unlink(tmp_path)
            st.stop()

Appending to C:\Users\82102\Desktop\ai_music/app.py


In [21]:
%%writefile -a "{NB_DIR}/app.py"

    col1, col2 = st.columns([1, 1])

    # ── col1: 예측 결과 ────────────────────────────────
    with col1:
        st.subheader("예측 결과")
        top_genre, top_prob = top3[0]
        emoji = GENRE_EMOJI.get(top_genre, "")
        st.metric(
            label="1위 장르",
            value=f"{emoji} {top_genre.upper()}",
            delta=f"확률 {top_prob:.1%}",
        )

        # 확률 막대그래프
        df_prob = pd.DataFrame(top3, columns=["장르", "확률"])
        fig_bar, ax_bar = plt.subplots(figsize=(5, 2.5))
        colors_bar = ["#2563EB", "#7C3AED", "#059669"]
        ax_bar.barh(
            df_prob["장르"], df_prob["확률"],
            color=colors_bar, edgecolor="white"
        )
        ax_bar.set_xlim(0, 1)
        ax_bar.set_xlabel("확률")
        ax_bar.set_title("Top-3 예측")
        for i, (_, row) in enumerate(df_prob.iterrows()):
            ax_bar.text(
                row["확률"] + 0.01,
                i, f"{row['확률']:.1%}",
                va="center", fontsize=9
            )
        fig_bar.tight_layout()
        st.pyplot(fig_bar)
        plt.close(fig_bar)

Appending to C:\Users\82102\Desktop\ai_music/app.py


In [22]:
%%writefile -a "{NB_DIR}/app.py"

    # ── col2: 멜스펙트로그램 ───────────────────────────
    with col2:
        st.subheader("멜스펙트로그램")
        fig_mel = plot_melspectrogram(
            tmp_path,
            title=f"{uploaded.name}"
        )
        st.pyplot(fig_mel)
        plt.close(fig_mel)

    # ── 풍선 효과: 1위 확률 50% 이상일 때 ─────────────
    if top_prob >= 0.50:
        st.balloons()
        st.success(f"{emoji} {top_genre.upper()} 장르로 자신 있게 예측했습니다!")
    else:
        st.warning("확률이 낮습니다 — 다른 장르와 혼용된 곡일 수 있습니다.")

    # 임시 파일 삭제
    os.unlink(tmp_path)

else:
    st.info("왼쪽에서 WAV 파일을 업로드하면 장르를 예측합니다.")
    # 사용 예시 이미지
    st.markdown("""
    ### 사용 방법
    1. `Browse files` 버튼 클릭
    2. WAV 파일 선택 (예: `jazz.00000.wav`)
    3. 자동으로 분석 → 장르 + 멜스펙트로그램 표시

    > 팁: `genres_original` 폴더의 샘플 WAV로 먼저 테스트해보세요.
    """)

Appending to C:\Users\82102\Desktop\ai_music/app.py


In [23]:
# ┌──────────────────────────────────────────────────────┐
# │ [검증] app.py 문법 검사 — SyntaxError 재발 방지      │
# │ [왜] nested f-string 따옴표 같은 문법 오류는 여기      │
# │      노트북 실행으로는 안 잡히고 `streamlit run`에서   │
# │      서버가 뜨는 순간에야 터진다 — 저장 직후 확인     │
# └──────────────────────────────────────────────────────┘
import ast

_app_path = NB_DIR / 'app.py'
_app_src = _app_path.read_text(encoding='utf-8')
try:
    ast.parse(_app_src)
    print(f'[SYNTAX OK] {_app_path.name} — {len(_app_src.splitlines())}줄, 문법 오류 없음')
except SyntaxError as e:
    raise SyntaxError(
        f'app.py 문법 오류! {e.msg} (line {e.lineno}) — 위 %%writefile 셀들을 다시 확인하세요.'
    ) from e


[SYNTAX OK] app.py — 236줄, 문법 오류 없음


In [24]:
# ┌──────────────────────────────────────────────────────┐
# │ [Step 2] streamlit run 실행 안내 + 파일 구조 확인   │
# │ (실제 streamlit run은 터미널에서 실행하세요)        │
# └──────────────────────────────────────────────────────┘

# ── 현재 NB_DIR(AI_Music 루트) 폴더 파일 목록 (6강 산출물 포함) ──
print(f'{NB_DIR}/ 폴더 내용:')
for p in sorted(NB_DIR.iterdir()):
    size = p.stat().st_size if p.is_file() else 0
    size_str = f'{size:,} B' if size < 1024 else f'{size/1024:.1f} KB'
    print(f'  {p.name:<45} {size_str if p.is_file() else "(폴더)"}')

print()
print('=' * 55)
print('  앱 실행 방법')
print('=' * 55)
print()
print('# 1. 이 노트북이 있는 폴더로 이동')
print(f'   cd "{NB_DIR}"')
print()
print('# 2. Streamlit 실행')
print('   streamlit run app.py')
print()
print('# 3. 브라우저에서 자동으로 열림')
print('   Local URL:  http://localhost:8501')
print('   Network URL: http://0.0.0.0:8501  (같은 Wi-Fi 내 공유 가능)')
print()
print('# 주의: model_rf.joblib, label_encoder.joblib, scaler.joblib')
print('#       세 파일이 app.py와 같은 폴더에 있어야 합니다.')


C:\Users\82102\Desktop\ai_music/ 폴더 내용:
  app.py                                        10.0 KB
  app_preview.py                                2.4 KB
  app_v2.py                                     12.0 KB
  d8_fallback_data                              (폴더)
  Data                                          (폴더)
  label_encoder.joblib                          557 B
  model_rf.joblib                               39149.9 KB
  packages.txt                                  554 B
  README.md                                     3.8 KB
  requirements.txt                              224 B
  scaler.joblib                                 1.9 KB
  강의_AI_Pair                                 (폴더)

  앱 실행 방법

# 1. 이 노트북이 있는 폴더로 이동
   cd "C:\Users\82102\Desktop\ai_music"

# 2. Streamlit 실행
   streamlit run app.py

# 3. 브라우저에서 자동으로 열림
   Local URL:  http://localhost:8501
   Network URL: http://0.0.0.0:8501  (같은 Wi-Fi 내 공유 가능)

# 주의: model_rf.joblib, label_encoder.joblib, scaler.joblib
#       세 파일

### 🔍 섹션 4 결과 해석

#### st.cache_resource vs st.cache_data 차이

| 데코레이터 | 용도 | 공유 방식 |
|------------|------|----------|
| `@st.cache_resource` | 모델, DB 연결 등 공유 자원 | 모든 사용자·세션이 **같은 객체 공유** |
| `@st.cache_data` | 데이터 변환 결과, API 응답 | 사용자별 **독립 복사본** |

> **왜 모델에 cache_resource인가?**
> RandomForest 모델 객체는 읽기 전용 — 여러 사용자가 동시에 써도 안전.
> `cache_data`는 캐시 항목마다 복사본을 만들어 메모리 낭비.

#### 실패 케이스 & 해결법

```python
# ❌ 자주 나오는 오류
# FileNotFoundError: model_rf.joblib 없음
# → 6강_Streamlit_장르예측앱(AI_Pair).ipynb 를 먼저 실행해 모델 저장

# ❌ librosa load 오류 (MP3 업로드)
# → st.file_uploader(type=['wav']) 로 WAV만 허용

# ❌ plt.show() 후 빈 화면
# → st.pyplot(fig) 사용, plt.close(fig) 로 메모리 해제
```

**→ 실무 결론:** Streamlit 앱 오류의 90%는 경로 문제(model 파일 위치) 또는 plt.show() 사용 실수입니다.


---
<a id='section-5'></a>
## 5. Streamlit Cloud 배포 예고 (8강)

### 지금은 로컬 — 8강에서는 인터넷으로

```
[지금 6강]                    [8강 목표]
로컬 터미널                  Streamlit Cloud
streamlit run app.py  →  push to GitHub → 자동 배포
http://localhost:8501        https://your-app.streamlit.app
```

배포를 위해 필요한 파일 (모두 AI_Music/ 안에 이미 있습니다):
```
AI_Music/
├── app.py
├── model_rf.joblib          ← Git LFS or HF Hub로
├── label_encoder.joblib
├── scaler.joblib
└── requirements.txt         ← 이것이 핵심
```


In [25]:
# ┌──────────────────────────────────────────────────────┐
# │ [Step 1] requirements.txt 생성 (Streamlit Cloud용)   │
# │ 입력: 없음   출력: requirements.txt                  │
# └──────────────────────────────────────────────────────┘

# 핀 버전을 박으면 재현성 보장, 하지만 Cloud 빌드가 느려질 수 있음
# 여기서는 최소 버전 지정 방식 사용
# torch/torchvision은 5·7·8강 노트북 실행에 필요합니다.
REQ = '''# AI_Music 강의(1강~8강) 전체 의존성
streamlit>=1.32.0
librosa>=0.10.0
scikit-learn>=1.3.0
joblib>=1.3.0
numpy>=1.24.0
pandas>=2.0.0
matplotlib>=3.7.0
seaborn>=0.12.0
pillow>=10.0.0
torch>=2.0.0
torchvision>=0.16.0
'''

req_path = NB_DIR / 'requirements.txt'
with open(req_path, 'w', encoding='utf-8') as f:
    f.write(REQ)

print(f'[저장 완료] {req_path}')
print()
print(REQ)

print('=' * 55)
print('  8강 배포 체크리스트 (미리보기)')
print('=' * 55)
steps = [
    'GitHub 레포 생성 + app.py, requirements.txt, joblib 파일 push',
    'share.streamlit.io 접속 → New app → 레포 연결',
    '자동 빌드 & 배포 (5~10분)',
    '공개 URL 공유: https://{your-app}.streamlit.app',
    '(선택) W&B Artifact로 모델 버전 관리',
]
for i, s in enumerate(steps, 1):
    print(f'  {i}. {s}')

print()
print('이 강좌 밖의 심화 과정 예고:')
print('  이 앱에 음성 인식(STT)을 붙이면, WAV 업로드만으로')
print('  장르 예측 + 가사 자동 추출까지 확장할 수 있습니다.')
print('  (LangChain/LangGraph 기반 에이전트 파이프라인 영역)')


[저장 완료] C:\Users\82102\Desktop\ai_music\requirements.txt

# AI_Music 강의(1강~8강) 전체 의존성
streamlit>=1.32.0
librosa>=0.10.0
scikit-learn>=1.3.0
joblib>=1.3.0
numpy>=1.24.0
pandas>=2.0.0
matplotlib>=3.7.0
seaborn>=0.12.0
pillow>=10.0.0
torch>=2.0.0
torchvision>=0.16.0

  8강 배포 체크리스트 (미리보기)
  1. GitHub 레포 생성 + app.py, requirements.txt, joblib 파일 push
  2. share.streamlit.io 접속 → New app → 레포 연결
  3. 자동 빌드 & 배포 (5~10분)
  4. 공개 URL 공유: https://{your-app}.streamlit.app
  5. (선택) W&B Artifact로 모델 버전 관리

이 강좌 밖의 심화 과정 예고:
  이 앱에 음성 인식(STT)을 붙이면, WAV 업로드만으로
  장르 예측 + 가사 자동 추출까지 확장할 수 있습니다.
  (LangChain/LangGraph 기반 에이전트 파이프라인 영역)


### 🔍 섹션 5 결과 해석

#### requirements.txt 작성 규칙

| 표기 | 의미 | 언제 사용 |
|------|------|----------|
| `librosa>=0.10.0` | 0.10.0 이상 최신 | 개발·강의 환경 (유연) |
| `librosa==0.10.1` | 정확히 이 버전만 | 프로덕션 배포 (재현성) |
| `librosa` | 아무 버전 | 지양 (호환성 불확실) |

> **8강에서 배포 시**: `pip freeze > requirements.txt` 로 현재 환경을 정확히 고정하는 것이 안전합니다.

#### 모델 파일 배포 문제

- `model_rf.joblib`은 수십~수백 MB → GitHub 100MB 한도 초과 가능
- 해결책:
  - **Git LFS**: 대용량 파일 별도 저장소
  - **HF Hub**: `huggingface_hub.upload_file()` → 앱에서 `hf_hub_download()`
  - **앱 시작 시 재학습**: 소규모 데이터면 가능 (비추천, 시간 소요)

**→ 실무 결론:** 모델 파일 < 50MB → GitHub 직접. 그 이상 → HF Hub 또는 S3 사용.


---
## 보너스 — 데모 WAV로 앱 흐름 재현

NewJeans-Attention.wav (또는 합성 WAV)로 최종 파이프라인을 한 번 더 실행합니다.
앱을 실행하기 전에 "결과가 어떻게 나오는지" 노트북에서 미리 확인합니다.


In [26]:
# ┌──────────────────────────────────────────────────────┐
# │ [보너스] 데모 WAV 전체 파이프라인 실행              │
# │ WAV → 피처 → 예측 → 멜스펙트로그램 한 화면에       │
# └──────────────────────────────────────────────────────┘

# 데모 파일 선택 (우선순위: NewJeans → genres_original → 합성)
if DEMO_AVAILABLE:
    demo_wav  = DEMO_WAV
    demo_name = 'NewJeans-Attention.wav'
    demo_true = '(실제 장르 미지정)'
elif AVAILABLE_WAVS:
    demo_wav  = AVAILABLE_WAVS[0]
    demo_name = demo_wav.name
    demo_true = demo_wav.parent.name
else:
    demo_wav  = _tmp.name
    demo_name = '합성 사인파 440Hz'
    demo_true = 'N/A'

print(f'데모 파일: {demo_name}')
print(f'실제 장르: {demo_true}')
print()

# 예측
top3_demo = predict_genre(demo_wav, rf_loaded, le_loaded, sc_loaded)

# 시각화 구성 (앱 UI와 동일한 2-column 레이아웃)
fig, (ax_bar, ax_mel) = plt.subplots(1, 2, figsize=(13, 4))

# 왼쪽: Top-3 확률 막대
genres_top3 = [t[0] for t in top3_demo]
probs_top3  = [t[1] for t in top3_demo]
bar_colors  = [COLORS['primary'], COLORS['secondary'], COLORS['accent']]
bars = ax_bar.barh(genres_top3, probs_top3, color=bar_colors, edgecolor='white', height=0.5)
ax_bar.set_xlim(0, 1.0)
ax_bar.set_xlabel('확률')
ax_bar.set_title(f'Top-3 예측 — {genres_top3[0].upper()} ({probs_top3[0]:.1%})', fontsize=12)
for bar, prob in zip(bars, probs_top3):
    ax_bar.text(prob + 0.01, bar.get_y() + bar.get_height()/2,
                f'{prob:.1%}', va='center', fontsize=10)

# 오른쪽: 멜스펙트로그램
y_audio, sr = librosa.load(str(demo_wav), sr=22050, mono=True, duration=10.0)
mel = librosa.feature.melspectrogram(y=y_audio, sr=sr, n_mels=128, fmax=8000)
mel_db = librosa.power_to_db(mel, ref=np.max)
img = librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel',
                                fmax=8000, ax=ax_mel, cmap='magma')
fig.colorbar(img, ax=ax_mel, format='%+2.0f dB')
ax_mel.set_title(f'멜스펙트로그램 — {demo_name}', fontsize=12)

fig.suptitle(f'[6강 데모] {demo_name}  |  예측: {genres_top3[0].upper()}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n예측 결과:')
for rank, (genre, prob) in enumerate(top3_demo, 1):
    bar = '█' * int(prob * 40)
    print(f'  {rank}위: {genre:<12} {prob:.3f}  {bar}')


데모 파일: NewJeans-Attention.wav
실제 장르: (실제 장르 미지정)




예측 결과:
  1위: reggae       0.140  █████
  2위: disco        0.140  █████
  3위: jazz         0.130  █████


### 🔍 데모 결과 해석 — 왜 reggae·pop·jazz가 정확히 0.130으로 동률일까?

위 데모(`NewJeans-Attention.wav`)는 top-3 확률이 모두 **0.130으로 정확히 동률**입니다. 바로 위 단위 테스트(jazz.00000.wav, 1위 확률 0.820)와 비교하면 확신의 크기 차이가 뚜렷합니다 — 이건 모델이 세 장르 사이에서 '헷갈린' 게 아니라, **10개 클래스 중 어느 것도 이기지 못하고 확률이 고르게 흩어졌다**는 신호입니다.

- **모델은 배운 것만 안다**: 이 RandomForest는 GTZAN 10개 장르(blues/classical/country/disco/hiphop/jazz/metal/pop/reggae/rock)로만 학습했습니다. **K-pop이라는 카테고리 자체를 본 적이 없습니다** — 입력이 학습 분포 밖(out-of-distribution)이면 모델은 새 카테고리를 만들지 못하고 '그나마 비슷해 보이는' 기존 카테고리 중 하나로 억지로 분류할 뿐입니다.
- **낮은 확률 자체가 신호**: 확신 있는 예측은 1위 확률이 0.8대까지 올라갑니다(위 jazz 테스트). 1위조차 0.130이라는 건 나머지 확률이 남은 7개 클래스에 흩어져 있다는 뜻 — 숫자로 '이 입력은 낯설다'가 드러난 사례입니다.
- **3초 고정 구간의 한계**: `extract_features`는 파일의 **첫 3초(인트로)**만 사용합니다. 후렴보다 장르 특징이 약한 인트로 구간이었다면 이 역시 낮은 확신에 기여했을 수 있습니다.

**→ 실무 결론:** 확률이 낮고 여러 클래스가 동률이면 '모델이 틀렸다'가 아니라 **'이 입력이 학습 분포 밖에 있다'는 신호로 읽어야 합니다.** 그래서 app.py는 `top_prob >= 0.50`일 때만 자신 있게 결과를 보여주고, 그 아래에서는 경고 문구로 낮은 확신을 사용자에게 그대로 알립니다.

> 🎲 **불확실성 참고**: RandomForest는 `random_state=42`를 고정했으므로 신경망과 달리 **완전히 결정론적**입니다 — 같은 CSV·같은 코드라면 몇 번을 다시 돌려도 정확도가 소수점 넷째 자리까지 똑같이 나와야 정상입니다(scikit-learn·NumPy 버전이 크게 다르면 아주 미세한 차이가 날 수 있습니다). 반면 위 데모 WAV 파이프라인이나 단위 테스트 셀은 **CSV가 아니라 실제 WAV를 잘라 오디오 피처를 새로 추출**하므로, `extract_features`의 `duration=3.0`이 정확히 어느 3초를 잘라내는지(파일 앞부분 고정)에 따라 예측이 달라질 수 있습니다 — 즉 "모델의 결정론"과 "입력 오디오 구간 선택의 임의성"은 서로 다른 이야기입니다.


---
<a id='section-fin'></a>
## 마무리 — 6강 요약표 + 퀴즈


In [27]:
# ┌──────────────────────────────────────────────────────┐
# │ [최종 셀] 6강 실습 완료 체크리스트                   │
# └──────────────────────────────────────────────────────┘
print('=' * 55)
print('  6강 실습 완료 체크리스트')
print('=' * 55)

checks = [
    ('RandomForest 재학습 + joblib 저장',       MODEL_PATH.exists()),
    ('LabelEncoder 저장',                       LE_PATH.exists()),
    ('StandardScaler 저장',                     SC_PATH.exists()),
    ('extract_features 함수 smoke test 통과',   _extract_ok),
    ('predict_genre 함수 top-3 반환 확인',      _predict_ok),
    ('plot_melspectrogram Figure 반환 확인',    _melspec_ok),
    ('장르별 단위 테스트 (1개 WAV씩)',           len(results_single) > 0),
    ('배치 테스트 (3개 WAV × 10장르)',           len(df_batch) > 0 if WAV_AVAILABLE else True),
    ('app.py 파일 저장',                        (NB_DIR / 'app.py').exists()),
    ('requirements.txt 생성',                  (NB_DIR / 'requirements.txt').exists()),
]

all_pass = True
for item, status in checks:
    mark = '✓' if status else '✗'
    if not status:
        all_pass = False
    print(f'  [{mark}] {item}')

print()
if all_pass:
    print('  모든 항목 완료! streamlit run app.py 로 앱을 실행하세요.')
else:
    print('  미완료 항목을 확인하고 해당 셀을 다시 실행하세요.')

print()
print('다음 수업: 7강 — 신뢰도 표시 + 멀티파일 비교 + ResNet-18 도입 준비')
print('심화 수업: 8강 — ResNet-18 Fine-tuning + Streamlit Cloud 배포')


  6강 실습 완료 체크리스트
  [✓] RandomForest 재학습 + joblib 저장
  [✓] LabelEncoder 저장
  [✓] StandardScaler 저장
  [✓] extract_features 함수 smoke test 통과
  [✓] predict_genre 함수 top-3 반환 확인
  [✓] plot_melspectrogram Figure 반환 확인
  [✓] 장르별 단위 테스트 (1개 WAV씩)
  [✓] 배치 테스트 (3개 WAV × 10장르)
  [✓] app.py 파일 저장
  [✓] requirements.txt 생성

  모든 항목 완료! streamlit run app.py 로 앱을 실행하세요.

다음 수업: 7강 — 신뢰도 표시 + 멀티파일 비교 + ResNet-18 도입 준비
심화 수업: 8강 — ResNet-18 Fine-tuning + Streamlit Cloud 배포


---
## 6강 배운 것 요약표

| 개념 | 핵심 한 줄 | 코드 |
|------|-----------|------|
| `joblib.dump` | Python 객체를 파일로 직렬화 | `joblib.dump(model, 'model.joblib')` |
| `joblib.load` | 파일에서 객체 복원 | `model = joblib.load('model.joblib')` |
| `extract_features` | WAV → 57개 피처 벡터 | `librosa` 피처 함수 조합 |
| `predict_genre` | 피처 → top-3 확률 | `model.predict_proba()` |
| `plot_melspectrogram` | WAV → Figure 반환 | `librosa.display.specshow()` |
| `st.cache_resource` | 모델 1회 로드 후 캐시 | `@st.cache_resource` 데코레이터 |
| `st.file_uploader` | 브라우저 파일 업로드 위젯 | `type=['wav']` 제한 |
| `st.pyplot(fig)` | matplotlib Figure를 Streamlit에 표시 | `plt.show()` 대신 |
| `requirements.txt` | 배포 의존성 명세 | `pip freeze` or 수동 작성 |
| 단위 테스트 | 앱 실행 전 함수 검증 | 노트북에서 먼저 |

---

## 미니 퀴즈 (5문항)

**Q1.** `joblib.dump(model, path)`와 `pickle.dump(model, f)`의 차이는?
a) joblib은 NumPy 배열을 더 효율적으로 직렬화한다
b) pickle은 더 빠르다
c) 차이 없다
d) joblib은 GPU 모델만 저장 가능하다

<details><summary>정답 보기</summary>

**a) joblib은 NumPy 배열을 더 효율적으로 직렬화한다**
RandomForest는 내부에 수많은 NumPy 배열을 가집니다. joblib은 이를 메모리 매핑으로 처리해 pickle보다 빠르고 파일 크기가 작습니다.

</details>

---

**Q2.** `StandardScaler`를 저장하지 않고 앱에서 `scaler = StandardScaler().fit(new_wav_features)`를 쓰면 어떤 문제가 생기는가?
a) 오류가 발생한다
b) 새 WAV 1개 데이터로 fit → 평균·분산이 의미 없어지고 예측 정확도 급락
c) 더 정확해진다
d) 아무 문제 없다

<details><summary>정답 보기</summary>

**b)** Scaler는 반드시 훈련 데이터 전체의 통계를 기억해야 합니다. 새 샘플 1개로 fit하면 그 샘플의 평균=0, 분산=1로만 변환되어 의미 없습니다. (바로 위 "확인해보기"에서 직접 본 스케일링 생략 사고와 같은 계열의 문제입니다.)

</details>

---

**Q3.** Streamlit 앱에서 `plt.show()` 대신 무엇을 써야 하는가?
a) `plt.savefig()`
b) `st.pyplot(fig)`
c) `st.image(fig)`
d) `fig.show()`

<details><summary>정답 보기</summary>

**b) `st.pyplot(fig)`**
Streamlit은 디스플레이 없는 서버 환경에서 실행됩니다. `plt.show()`는 로컬 GUI를 열려 하지만 서버에는 GUI가 없습니다. `st.pyplot(fig)`는 Figure를 PNG로 변환해 브라우저에 전송합니다.

</details>

---

**Q4.** `@st.cache_resource`와 `@st.cache_data`의 차이로 올바른 것은?
a) cache_resource는 사용자별 독립 복사본, cache_data는 공유 객체
b) cache_resource는 공유 객체(모델·DB), cache_data는 사용자별 독립 복사본(데이터 변환)
c) 둘은 동일하다
d) cache_data가 더 빠르다

<details><summary>정답 보기</summary>

**b)** 모델처럼 읽기 전용 공유 자원은 `cache_resource`. 사용자 입력에 따라 달라지는 변환 결과는 `cache_data`.

</details>

---

**Q5.** `extract_features()` 함수가 `FEATURE_COLS` 순서대로 벡터를 만드는 이유는?
a) librosa 함수 호출 순서가 정해져 있어서
b) RandomForest가 피처 이름을 기억하기 때문에
c) 학습 시 DataFrame 컬럼 순서와 예측 시 배열 순서가 일치해야 하기 때문에
d) 상관없다

<details><summary>정답 보기</summary>

**c)** scikit-learn 모델은 학습 때 본 피처 순서를 그대로 기대합니다. 순서가 바뀌면 'tempo'가 'chroma_stft_mean' 자리에 들어가는 치명적 오류가 발생합니다. `FEATURE_COLS` 리스트를 통해 순서를 고정하는 것이 핵심 설계 결정입니다.

</details>


---
## 🤝 AI Pair 섹션 — 모델 저장·서빙·앱 배포

> **목표**: AI를 답 베끼는 도구가 아니라 *내 코드·판단을 검증해주는 동료*로 쓴다. (읽기용 — 실습 시간엔 핵심만 따라가도 됩니다)

| 단계 | 내가 하는 것 | AI가 하는 것 |
|---|---|---|
| 1️⃣ Solo | 먼저 직접 작성/판단 | (아직 X) |
| 2️⃣ Review | 내 코드·판단을 제출 | 리뷰 + 근거 설명 |
| 3️⃣ Debug | AI가 준 "조용히 틀린" 코드의 결함 찾기 | 의도적 버그 제공 |
| 4️⃣ Prompt Card | 실험 설계 프롬프트 익히기 | — |

### 1️⃣ Solo — 먼저 스스로 풀어보세요
💡 *AI에게 물으면 30초 만에 답이 나오지만, 지금 막히는 지점이 이해의 핵심입니다.*


### ✏️ [Solo 레벨 1] 직접 작성해 보세요 — extract_features의 duration을 3.0 → 5.0으로 바꾸기

힌트: extract_features 함수를 통째로 다시 정의하되, librosa.load(..., duration=3.0)만 duration=5.0으로 바꾸세요.
그 다음, 이미 저장된 rf_loaded/sc_loaded(3초 기준으로 학습됨)에 5초짜리 벡터를 넣어 predict_genre를 호출해보세요.
질문: 에러 없이 실행될까요? 정확도는 3초 버전과 비슷할까요, 달라질까요? 왜 그럴지 먼저 적어본 뒤 실행하세요.

In [28]:
# TODO: 여기에 작성하세요

### ✏️ [Solo 레벨 2] 직접 작성해 보세요 — predict_genre가 top-3 대신 top-5를 반환하도록 수정

힌트: predict_genre 함수를 복사해 이름을 predict_genre_top5로 바꾸고,
top3_idx = np.argsort(proba)[::-1][:3] 부분의 슬라이스 개수만 바꾸세요.
먼저 예측: 4~5위 장르의 확률은 1~3위보다 뚜렷이 낮을까요, 비슷하게 낮을까요? 예측을 먼저 적어본 뒤 실행하세요.
확인: jazz.00000.wav에 대해 4~5위 장르와 확률도 함께 출력해보세요.

In [29]:
# TODO: 여기에 작성하세요

### ✏️ [Solo 레벨 3] 직접 작성해 보세요 — N_PER_GENRE를 바꿔가며 정확도 추이 실험

힌트:
1) def evaluate_batch(n_per_genre): ... 함수를 만드세요 (위 [Step 2] 배치 테스트 코드를 재사용).
2) [1, 3, 5, 10] 각각에 대해 evaluate_batch를 호출해 전체 정확도를 딕셔너리에 저장하세요.
3) matplotlib으로 x=n_per_genre, y=정확도 그래프를 그려, 테스트 샘플 수가 늘수록 정확도 추정이
얼마나 안정되는지(널뛰기가 줄어드는지) 눈으로 확인하세요.

In [30]:
def evaluate_batch(n_per_genre):
    # TODO: 여기에 작성하세요
    pass

# evaluate_batch 결과를 리스트에 모아 그래프로 그려보세요.


### 2️⃣ Review
아래 프롬프트를 복사해 ChatGPT/Claude에 붙여넣으세요.
```text
음악 장르 예측 파이프라인입니다: librosa로 WAV에서 57개 피처(chroma·rms·spectral·mfcc 등) 추출 →
StandardScaler.transform → RandomForestClassifier.predict_proba → top-3 장르.
모델/encoder/scaler는 joblib으로 저장해 Streamlit 앱이 불러옵니다.

내가 extract_features의 duration을 3.0 → 5.0으로 바꿔서 실행했더니 예측 확률이 흔들렸습니다.
1) scaler.transform이 왜 "학습 때와 같은 분포"를 가정하는지, 2) duration을 바꾸면 어떤 피처들이
가장 크게 흔들릴지(tempo? mfcc? rms?), 3) 이걸 안전하게 바꾸려면 무엇을 다시 해야 하는지
(힌트: 재학습) 각각 근거와 함께 설명해줘.
```


In [31]:
# ── 2️⃣ Review (코드형) — 로컬/클라우드 LLM에게 내 코드·판단을 리뷰받기 ──
# [사전조건] 로컬: LM Studio(00-1)/Ollama(00-2) 서버 실행  |  클라우드: OPENROUTER/OPENAI 키 설정
# openai 패키지가 없으면 아래 주석을 풀어 한 번만 실행하세요 (이미 있으면 건너뛰기)
# !pip install openai
import os
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

if OpenAI is None:
    print("[안내] openai 패키지가 없어 이 Review 셀을 건너뜁니다 — `!pip install openai` 후 다시 실행하세요(선택 사항입니다).")
else:
    PROVIDER = "lmstudio"   # "lmstudio" | "ollama" | "openrouter" | "openai"  ← 한 줄만 바꾸면 전환
    PROVIDERS = {
        "lmstudio":   {"base_url": "http://localhost:1234/v1",  "model": "local-model",          "api_key": "lm-studio"},
        "ollama":     {"base_url": "http://localhost:11434/v1", "model": "llama3.2",              "api_key": "ollama"},
        "openrouter": {"base_url": "https://openrouter.ai/api/v1", "model": "anthropic/claude-3.5-sonnet", "api_key": os.getenv("OPENROUTER_API_KEY", "")},
        "openai":     {"base_url": "https://api.openai.com/v1", "model": "gpt-4o-mini",           "api_key": os.getenv("OPENAI_API_KEY", "")},
    }
    cfg = PROVIDERS[PROVIDER]
    client = OpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"])

    review_prompt = '''위 markdown 셀의 Review 프롬프트를 그대로 붙여넣어 실제 API 호출로 리뷰를 받아보세요.
    (강의 중에는 시간 관계상 markdown 프롬프트만 복사해 웹 UI에 붙여넣는 방식을 권장합니다.)'''

    try:
        resp = client.chat.completions.create(
            model=cfg["model"],
            messages=[{"role": "user", "content": review_prompt}],
            timeout=20,
        )
        print(resp.choices[0].message.content)
    except Exception as e:
        print(f'[안내] {PROVIDER} 연결 실패 — 로컬 서버 실행 여부 또는 API 키를 확인하세요: {e}')


[안내] lmstudio 연결 실패 — 로컬 서버 실행 여부 또는 API 키를 확인하세요: Connection error.


### 3️⃣ Debug — "조용히 틀린" 코드 찾기
아래는 **실행은 되고 에러도 안 나지만, 조용히 잘못 예측하는** 코드입니다. 결함을 찾아 고치세요.


In [32]:
# [Debug] 아래 코드는 에러 없이 실행됩니다 — 그런데 조용히 잘못 예측합니다. 결함을 찾아 고쳐보세요.

def predict_genre_buggy(wav_path, model, label_encoder, scaler):
    # 1. 피처 추출
    vec = extract_features(wav_path)          # shape (1, 57)

    # 2. ⚠️ 여기를 의심해보세요 — scaler를 거치지 않고 원본 벡터를 그대로 씁니다
    proba = model.predict_proba(vec)[0]        # 정상 코드는 scaler.transform(vec)을 먼저 거칩니다

    # 3. top-3 인덱스
    top3_idx = np.argsort(proba)[::-1][:3]
    return [(label_encoder.classes_[i], float(proba[i])) for i in top3_idx]


if AVAILABLE_WAVS:
    _test_wav = AVAILABLE_WAVS[0]
    _buggy_result = predict_genre_buggy(_test_wav, rf_loaded, le_loaded, sc_loaded)
    print(f'[버그 버전] 예측 결과 ({_test_wav.name}):')
    for rank, (genre, prob) in enumerate(_buggy_result, 1):
        print(f'  {rank}위: {genre:<12} {prob:.3f}')
else:
    print('[WAV 없음] Debug 셀은 genres_original 데이터가 있어야 의미 있게 비교됩니다.')


[버그 버전] 예측 결과 (classical.00000.wav):
  1위: hiphop       0.220
  2위: reggae       0.170
  3위: disco        0.160


💭 **생각해 볼 점**:
- 이 코드는 shape 에러 없이 잘 "실행"됩니다. 왜 에러가 안 날까요? (`scaler.transform()`을 거치든 안 거치든 벡터의 shape은 `(1, 57)`로 동일하기 때문입니다 — RandomForest는 shape만 맞으면 값의 스케일이 뭐든 일단 예측을 "내놓기는" 합니다.)
- RandomForest는 학습할 때 `X_tr_sc`(스케일링된 값)의 분포를 기준으로 나뭇가지 분기 기준(threshold)을 정했습니다. 스케일링 안 된 원본 값을 넣으면 그 기준들이 전혀 다른 값 범위에 적용되는 셈인데, 예측 확률·1위 장르가 어떻게 흔들릴까요?
- 위 셀을 직접 실행해서 나온 top-3와, 몇 셀 앞의 "확인해보기"에서 계산한 `acc_scaled` vs `acc_unscaled` 정확도 차이를 연결해서 설명해보세요 — 지금 이 debug 함수가 바로 그 "acc_unscaled" 상황을 함수 하나로 재현한 것입니다.


### 4️⃣ Prompt Card
📝 **카드 1~3** (이 단원 최적화, 실험 설계형):
```text
1. "extract_features의 duration을 3.0에서 1.0으로 줄이면 예측 정확도와 처리 속도가 어떻게 달라질지 먼저 예측한 뒤 AI에게 근거를 묻고, 실행해서 확인해줘."
2. "n_estimators를 100 → 300으로 늘리면 학습 시간과 정확도가 각각 어떻게 달라질지 예측한 뒤 AI에게 이유를 물어봐줘."
3. "predict_genre가 top-3 대신 확률이 threshold(예: 0.15) 이상인 장르만 가변 개수로 반환하도록 바꾸면 어떤 장점·단점이 있을지 AI와 같이 설계하고 구현해줘."
```
🎯 **마무리 체크**: [ ] Solo 직접 [ ] Review 실행검증 [ ] Debug 결함 찾음 [ ] Prompt Card 1개 내 노트에 정리


---
## 📚 [세션 요약] Streamlit으로 장르 예측 앱 만들기

> 🎯 **핵심 Takeaways**
> 1. **이론적 근거**: `joblib.dump/load`로 모델을 파일 경계 너머로 옮기면, "학습된 지식"과 "그 지식을 쓰는 프로그램"을 분리할 수 있다 — 이 분리가 모델 서빙(model serving)의 시작점.
> 2. **실무적 활용**: 노트북에서 함수로 먼저 검증한 뒤 그대로 `app.py`에 옮기면, UI 버그와 모델 버그를 구분해 디버깅할 수 있다. 스케일링 누락처럼 "에러 없이 조용히 틀리는" 실수가 가장 위험하다.
>
> ➡️ **다음 단계**: `7강_Streamlit고급_ResNet도입(AI_Pair).ipynb` — 오늘 만든 앱에 예측 신뢰도 표시와 멀티파일 비교 기능을 더하고, ResNet-18 도입을 준비합니다.
